# LeetCode 1234 — Replace the Substring for Balanced String bằng Python

Link đề: https://leetcode.com/problems/replace-the-substring-for-balanced-string/ · Độ khó: **Medium** · Chủ đề: String, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def balancedString(self, s: str) -> int:
        ...
```

Ghi chú: trả về **độ dài nhỏ nhất** của một xâu con liên tiếp cần thay thế (bằng một xâu bất kỳ cùng độ dài) để `s` cân bằng; trả `0` nếu `s` đã cân bằng.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Biến điều kiện "thay được để cân bằng" thành điều kiện kiểm tra đơn giản: **mọi ký tự nằm ngoài cửa sổ xuất hiện không quá `n/4` lần**.
2. Đi từ **thử mọi xâu con** `O(n³)` → **mảng đếm tiền tố (prefix count)** `O(n²)` → **tìm kiếm nhị phân trên độ dài + cửa sổ cố định** `O(n log n)` → **cửa sổ trượt hai con trỏ (sliding window)** `O(n)`.
3. Hiểu tính **đơn điệu**: nếu cửa sổ `[i, j]` thoả thì mọi cửa sổ chứa nó cũng thoả — điều kiện để dùng hai con trỏ và nhị phân.
4. Cài đặt khuôn mẫu "**cửa sổ ngắn nhất thoả điều kiện**": mở rộng phải, co trái chừng nào còn thoả.
5. Tránh các lỗi: nghĩ đáp án = tổng số ký tự dư, đếm trong cửa sổ thay vì ngoài cửa sổ, quên hoàn lại đếm khi co trái, quên trường hợp đã cân bằng.

## 2. Đề bài & ví dụ

Cho xâu `s` độ dài `n` chỉ gồm bốn loại ký tự `'Q'`, `'W'`, `'E'`, `'R'`.

Xâu được gọi là **cân bằng** nếu mỗi ký tự xuất hiện đúng `n / 4` lần.

Trả về **độ dài nhỏ nhất** của một xâu con (liên tiếp) có thể được thay bằng **một xâu bất kỳ cùng độ dài** để `s` trở nên cân bằng. Nếu `s` đã cân bằng, trả về `0`.

**Ví dụ 1:** `s = "QWER"` → `0` (đã cân bằng).

**Ví dụ 2:** `s = "QQWE"` → `1` (thay một `'Q'` bằng `'R'`: `"RQWE"` hoặc `"QRWE"`).

**Ví dụ 3:** `s = "QQQW"` → `2` (thay `"QQ"` đầu bằng `"ER"`).

**Ràng buộc:**
- `n == s.length`, `4 <= n <= 10^5`
- `n` chia hết cho 4.
- `s` chỉ chứa `'Q'`, `'W'`, `'E'`, `'R'`.

In [1]:
import random
import time
from collections import Counter
from typing import List

examples = [("QWER", 0), ("QQWE", 1), ("QQQW", 2), ("QQQQ", 3)]
for s, want in examples:
    print(f"{s!r} -> {want}")

'QWER' -> 0
'QQWE' -> 1
'QQQW' -> 2
'QQQQ' -> 3


(Thêm ví dụ `"QQQQ"` → `3`: phải thay 3 chữ `Q` liền nhau thành `W, E, R`.)

## 3. Kiến thức nền

### 3.1 `Counter` — đếm tần suất

`collections.Counter(s)` cho số lần xuất hiện của từng ký tự. Truy cập ký tự không có trả `0` (không lỗi).

In [2]:
cnt = Counter("QQWE")
print(cnt, cnt["Q"], cnt["R"])

Counter({'Q': 2, 'W': 1, 'E': 1}) 2 0


### 3.2 Xâu con và "phần bên ngoài"

Xâu con `s[i:j]` (từ chỉ số `i` đến `j−1`). Phần **bên ngoài** là `s[:i] + s[j:]`. Đếm bên ngoài = đếm toàn bộ − đếm bên trong.

In [3]:
s = "QQWEQR"
i, j = 1, 4
inside, outside = s[i:j], s[:i] + s[j:]
print("trong:", inside, Counter(inside))
print("ngoài:", outside, Counter(outside))
print("toàn bộ - trong == ngoài?", Counter(s) - Counter(inside) == Counter(outside))

trong: QWE Counter({'Q': 1, 'W': 1, 'E': 1})
ngoài: QQR Counter({'Q': 2, 'R': 1})
toàn bộ - trong == ngoài? True


### 3.3 Mảng đếm tiền tố (prefix count)

`pre[k][c]` = số lần ký tự `c` xuất hiện trong `s[:k]`. Khi đó số lần `c` trong `s[i:j]` là `pre[j][c] − pre[i][c]` — tính trong `O(1)` thay vì `O(độ dài)`.

In [4]:
IDX = {"Q": 0, "W": 1, "E": 2, "R": 3}


def build_prefix(s):
    pre = [[0, 0, 0, 0]]
    for ch in s:
        row = pre[-1][:]
        row[IDX[ch]] += 1
        pre.append(row)
    return pre


pre = build_prefix("QQWEQR")
print(pre)
print("số Q trong s[1:4]:", pre[4][0] - pre[1][0])

[[0, 0, 0, 0], [1, 0, 0, 0], [2, 0, 0, 0], [2, 1, 0, 0], [2, 1, 1, 0], [3, 1, 1, 0], [3, 1, 1, 1]]
số Q trong s[1:4]: 1


### 3.4 Hai con trỏ / cửa sổ trượt

Cửa sổ `[left, right]` di chuyển trên mảng: `right` tiến để **mở rộng**, `left` tiến để **co lại**. Mỗi con trỏ chỉ đi từ trái sang phải, tổng cộng `O(n)` bước. Kỹ thuật hợp lệ khi điều kiện có tính **đơn điệu** (mở rộng không làm mất điều kiện).

In [5]:
# Ví dụ nhỏ: đoạn ngắn nhất có tổng >= 7 (các số dương)
a = [2, 3, 1, 2, 4, 3]
best, left, total = len(a) + 1, 0, 0
for right, x in enumerate(a):
    total += x
    while total >= 7:
        best = min(best, right - left + 1)
        total -= a[left]
        left += 1
print("đoạn ngắn nhất tổng >= 7 dài:", best)

đoạn ngắn nhất tổng >= 7 dài: 2


## 4. Ý tưởng nền tảng / trực giác

Gọi `k = n / 4`. Ta được thay xâu con `s[i:j]` bằng **bất cứ thứ gì**. Vậy câu hỏi: khi nào việc thay đó **có thể** làm `s` cân bằng?

- Các ký tự **bên ngoài** cửa sổ giữ nguyên. Nếu bên ngoài đã có quá `k` chữ `Q` thì vô vọng — thay bên trong không bớt được `Q` bên ngoài.
- Ngược lại, nếu mọi ký tự bên ngoài đều `≤ k`, ta điền vào cửa sổ đúng `k − out[c]` chữ `c` cho mỗi `c`. Tổng số cần điền = `4k − (tổng bên ngoài) = n − (n − L) = L` — **vừa khít** độ dài cửa sổ!

> **Cửa sổ `s[i:j]` hợp lệ ⇔ với mọi `c ∈ {Q, W, E, R}`: `out[c] ≤ n/4`.**

Tính tay `"QQQW"` (`k = 1`): toàn bộ `Q=3, W=1`. Cửa sổ phải "ôm" đủ số `Q` dư để bên ngoài còn `≤ 1` chữ `Q` → cửa sổ chứa ≥ 2 chữ `Q` → ngắn nhất là `"QQ"` (độ dài 2).

In [6]:
def valid_window(s, i, j):
    k = len(s) // 4
    out = Counter(s[:i] + s[j:])
    return all(out[c] <= k for c in "QWER")


s = "QQQW"
for i, j in [(0, 1), (0, 2), (1, 3), (2, 4)]:
    print(f"s[{i}:{j}] = {s[i:j]!r:6} hợp lệ? {valid_window(s, i, j)}")

s[0:1] = 'Q'    hợp lệ? False
s[0:2] = 'QQ'   hợp lệ? True
s[1:3] = 'QQ'   hợp lệ? True
s[2:4] = 'QW'   hợp lệ? False


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** thử mọi cặp `(i, j)` với `0 ≤ i ≤ j ≤ n`, kiểm tra `valid_window`, lấy độ dài nhỏ nhất. Cửa sổ rỗng (`i == j`) hợp lệ khi `s` đã cân bằng → trả `0` tự nhiên.

In [7]:
def balanced_brute(s):
    n = len(s)
    best = n
    for i in range(n + 1):
        for j in range(i, n + 1):
            if j - i < best and valid_window(s, i, j):
                best = j - i
    return best


for s, want in examples:
    got = balanced_brute(s)
    print(s, got)
    assert got == want

QWER 0
QQWE 1
QQQW 2
QQQQ 3


**Độ phức tạp:** `O(n²)` cửa sổ, mỗi lần đếm lại `O(n)` → Time `O(n³)`. Với `n = 10^5` là `10^15` — vô vọng. Space `O(n)`.

**Nút thắt:** (1) đếm lại từ đầu cho mỗi cửa sổ; (2) thử cả những cửa sổ chắc chắn không tốt hơn.

## 6. Bước đột phá

**Tính đơn điệu:** nếu cửa sổ `[i, j)` hợp lệ thì mọi cửa sổ **chứa** nó cũng hợp lệ (mở rộng chỉ làm `out[c]` **giảm** hoặc giữ nguyên). Ngược lại, nếu `[i, j)` không hợp lệ thì mọi cửa sổ **con** của nó cũng không hợp lệ.

Hệ quả:
1. **Với mỗi `left`, `right` nhỏ nhất làm cửa sổ hợp lệ không giảm khi `left` tăng** → hai con trỏ cùng chiều, `O(n)`.
2. **"Có cửa sổ hợp lệ độ dài `L` không?" đơn điệu theo `L`** → tìm kiếm nhị phân trên `L`.

**Quy tắc cửa sổ trượt (dạng "ngắn nhất thoả"):**
- Đếm toàn bộ vào `out` (coi như cửa sổ rỗng, mọi ký tự ở ngoài).
- Với mỗi `right`: đưa `s[right]` vào cửa sổ → `out[s[right]] -= 1`.
- **Chừng nào** cửa sổ `[left, right]` hợp lệ (`max(out) ≤ k`) **thì** cập nhật đáp án với `right − left + 1`, rồi đẩy `s[left]` ra → `out[s[left]] += 1`, `left += 1`.

**Bất biến:** sau vòng `while`, cửa sổ `[left, right]` **không** hợp lệ (hoặc rỗng), và mọi cửa sổ kết thúc tại `right` có đầu trái `< left` đã được xét.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Prefix count + thử mọi cặp, `O(n²)`

In [8]:
def balanced_prefix_n2(s):
    n, k = len(s), len(s) // 4
    pre = build_prefix(s)
    total = pre[n]
    best = n
    for i in range(n + 1):
        for j in range(i, n + 1):
            if j - i >= best:
                break
            if all(total[c] - (pre[j][c] - pre[i][c]) <= k for c in range(4)):
                best = j - i
                break                      # j nhỏ nhất cho i này
    return best


for s, want in examples:
    assert balanced_prefix_n2(s) == want
print("Cách 2 đúng trên ví dụ")

Cách 2 đúng trên ví dụ


Với mỗi `i`, ta tăng `j` và **dừng ngay** ở `j` hợp lệ đầu tiên (nhờ đơn điệu, `j` lớn hơn chỉ dài hơn) hoặc khi độ dài đã `≥ best`. **Độ phức tạp:** Time `O(n²)` tệ nhất, Space `O(n)`.

### 7.2 Cách 3 — Tìm kiếm nhị phân trên độ dài + cửa sổ cố định, `O(n log n)`

`can(L)` = có cửa sổ độ dài `L` hợp lệ không. Kiểm tra bằng cửa sổ trượt **cố định** độ dài `L` trong `O(n)`. `can` đơn điệu: `can(L)` đúng ⇒ `can(L+1)` đúng (mở rộng cửa sổ hợp lệ thêm 1 ô — nếu `L < n`).

In [9]:
def balanced_binary_search(s):
    n, k = len(s), len(s) // 4
    total = Counter(s)

    def can(L):
        out = dict(total)
        for c in s[:L]:
            out[c] -= 1
        if all(out.get(c, 0) <= k for c in "QWER"):
            return True
        for r in range(L, n):              # trượt cửa sổ [r-L+1, r]
            out[s[r]] -= 1
            out[s[r - L]] += 1
            if all(out.get(c, 0) <= k for c in "QWER"):
                return True
        return False

    lo, hi = 0, n                          # can(n) luôn đúng
    while lo < hi:
        mid = (lo + hi) // 2
        if can(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


for s, want in examples:
    assert balanced_binary_search(s) == want
print("Cách 3 đúng trên ví dụ")

Cách 3 đúng trên ví dụ


**Độ phức tạp:** Time `O(n log n)` (mỗi `can` là `O(n)`, `log n` lần), Space `O(1)` ngoài đếm.

### 7.3 Cách 4 — Cửa sổ trượt hai con trỏ, `O(n)` (lời giải chuẩn)

In [10]:
def balanced_sliding(s):
    n, k = len(s), len(s) // 4
    out = Counter(s)
    if all(out[c] <= k for c in "QWER"):
        return 0
    best, left = n, 0
    for right, ch in enumerate(s):
        out[ch] -= 1                                   # s[right] vào cửa sổ
        while left <= right and all(out[c] <= k for c in "QWER"):
            best = min(best, right - left + 1)
            out[s[left]] += 1                          # s[left] ra khỏi cửa sổ
            left += 1
    return best


for s, want in examples:
    assert balanced_sliding(s) == want
print("Cách 4 đúng trên ví dụ")

Cách 4 đúng trên ví dụ


**Độ phức tạp:** mỗi chỉ số vào cửa sổ một lần và ra tối đa một lần; mỗi lần kiểm tra `all(...)` là `O(4)`. Time `O(n)`, Space `O(1)` (4 bộ đếm).

### 7.4 Cách 5 — Chỉ theo dõi "lượng dư" cần ôm

Một góc nhìn tương đương: gọi `need[c] = max(0, total[c] − k)` là số chữ `c` **dư**. Cửa sổ hợp lệ ⇔ chứa **ít nhất** `need[c]` chữ `c` với mọi `c`. Đây chính là dạng "**Minimum Window Substring**" (LeetCode 76): tìm cửa sổ ngắn nhất chứa đủ một "đa tập" ký tự. Dùng biến `missing` = tổng số ký tự còn thiếu để kiểm tra trong `O(1)`.

In [11]:
def balanced_need(s):
    n, k = len(s), len(s) // 4
    total = Counter(s)
    need = {c: max(0, total[c] - k) for c in "QWER"}
    missing = sum(need.values())
    if missing == 0:
        return 0
    have = {c: 0 for c in "QWER"}
    best, left = n, 0
    for right, ch in enumerate(s):
        have[ch] += 1
        if have[ch] <= need[ch]:
            missing -= 1                                # vừa ôm thêm một ký tự còn thiếu
        while missing == 0:
            best = min(best, right - left + 1)
            lc = s[left]
            have[lc] -= 1
            if have[lc] < need[lc]:
                missing += 1
            left += 1
    return best


for s, want in examples:
    assert balanced_need(s) == want
print("Cách 5 đúng trên ví dụ; need cho 'QQQW':", {c: max(0, Counter('QQQW')[c] - 1) for c in 'QWER'})

Cách 5 đúng trên ví dụ; need cho 'QQQW': {'Q': 2, 'W': 0, 'E': 0, 'R': 0}


**Độ phức tạp:** Time `O(n)`, Space `O(1)`. Lợi ích: điều kiện kiểm tra `missing == 0` là `O(1)` và khuôn mẫu dùng lại được cho bảng chữ cái lớn.

## 8. TRACE từng bước

Trace Cách 4 với `s = "QQQW"` (`k = 1`). Ban đầu `out = {Q:3, W:1, E:0, R:0}` — không cân bằng.

| right | ch | out sau khi đưa vào | hợp lệ? | Hành động trong while | best | left |
|---|---|---|---|---|---|---|
| 0 | Q | Q2 W1 | không (Q=2) | — | 4 | 0 |
| 1 | Q | Q1 W1 | **có** | best = 2; đẩy Q ra → Q2 | 2 | 1 |
| 2 | Q | Q1 W1 | **có** | best = min(2, 2); đẩy Q ra → Q2 | 2 | 2 |
| 3 | W | Q2 W0 | không | — | 2 | 2 |

Kết quả `2`.

In [12]:
def trace_sliding(s):
    n, k = len(s), len(s) // 4
    out = Counter(s)
    show = lambda: " ".join(f"{c}{out[c]}" for c in "QWER")
    print(f"k = {k}, out ban đầu: {show()}")
    if all(out[c] <= k for c in "QWER"):
        print("đã cân bằng -> 0")
        return 0
    best, left = n, 0
    for right, ch in enumerate(s):
        out[ch] -= 1
        print(f"right={right} đưa {ch!r} vào -> out: {show()}")
        while left <= right and all(out[c] <= k for c in "QWER"):
            best = min(best, right - left + 1)
            print(f"   cửa sổ [{left},{right}]={s[left:right + 1]!r} hợp lệ, best={best}; đẩy {s[left]!r} ra")
            out[s[left]] += 1
            left += 1
    print("kết quả:", best)
    return best


trace_sliding("QQQW")
print()
trace_sliding("WQWRQQQW")

k = 1, out ban đầu: Q3 W1 E0 R0
right=0 đưa 'Q' vào -> out: Q2 W1 E0 R0
right=1 đưa 'Q' vào -> out: Q1 W1 E0 R0
   cửa sổ [0,1]='QQ' hợp lệ, best=2; đẩy 'Q' ra
right=2 đưa 'Q' vào -> out: Q1 W1 E0 R0
   cửa sổ [1,2]='QQ' hợp lệ, best=2; đẩy 'Q' ra
right=3 đưa 'W' vào -> out: Q2 W0 E0 R0
kết quả: 2

k = 2, out ban đầu: Q4 W3 E0 R1
right=0 đưa 'W' vào -> out: Q4 W2 E0 R1
right=1 đưa 'Q' vào -> out: Q3 W2 E0 R1
right=2 đưa 'W' vào -> out: Q3 W1 E0 R1
right=3 đưa 'R' vào -> out: Q3 W1 E0 R0
right=4 đưa 'Q' vào -> out: Q2 W1 E0 R0
   cửa sổ [0,4]='WQWRQ' hợp lệ, best=5; đẩy 'W' ra
   cửa sổ [1,4]='QWRQ' hợp lệ, best=4; đẩy 'Q' ra
right=5 đưa 'Q' vào -> out: Q2 W2 E0 R0
   cửa sổ [2,5]='WRQQ' hợp lệ, best=4; đẩy 'W' ra
right=6 đưa 'Q' vào -> out: Q1 W3 E0 R0
right=7 đưa 'W' vào -> out: Q1 W2 E0 R0
   cửa sổ [3,7]='RQQQW' hợp lệ, best=4; đẩy 'R' ra
   cửa sổ [4,7]='QQQW' hợp lệ, best=4; đẩy 'Q' ra
   cửa sổ [5,7]='QQW' hợp lệ, best=3; đẩy 'Q' ra
kết quả: 3


3

## 9. Code nộp LeetCode

In [13]:
class Solution:
    def balancedString(self, s: str) -> int:
        n = len(s)
        k = n // 4
        out = Counter(s)
        if all(out[c] <= k for c in "QWER"):
            return 0
        ans, left = n, 0
        for right, ch in enumerate(s):
            out[ch] -= 1
            while left <= right and out["Q"] <= k and out["W"] <= k and out["E"] <= k and out["R"] <= k:
                ans = min(ans, right - left + 1)
                out[s[left]] += 1
                left += 1
        return ans


sol = Solution()
for s, want in examples:
    got = sol.balancedString(s)
    print(s, got)
    assert got == want

QWER 0
QQWE 1
QQQW 2
QQQQ 3


## 10. Vì sao đúng

**Bổ đề 1 (điều kiện hợp lệ):** thay được `s[i:j]` để cân bằng ⇔ `out[c] ≤ k` với mọi `c`. (⇒) sau khi thay, mỗi ký tự có đúng `k` lần, trong đó phần bên ngoài không đổi nên `out[c] ≤ k`. (⇐) điền `k − out[c] ≥ 0` chữ `c` vào cửa sổ; tổng `4k − Σ out = n − (n − L) = L`, vừa khít.

**Bổ đề 2 (đơn điệu):** mở rộng cửa sổ chỉ làm `out[c]` giảm → hợp lệ vẫn hợp lệ.

**Đúng đắn của hai con trỏ:** gọi `L*` là đáp án, đạt bởi cửa sổ `[i*, j*]`. Khi `right` chạy tới `j*`, xét `left`:
- Nếu `left ≤ i*`: cửa sổ `[left, j*]` chứa `[i*, j*]` nên hợp lệ (Bổ đề 2); vòng `while` sẽ co `left` tới ít nhất `i*`... và ghi nhận độ dài `j* − i* + 1 = L*` khi `left = i*` (vì `[i*, j*]` hợp lệ).
- `left > i*` là không thể: `left` chỉ vượt qua `i*` khi cửa sổ `[i*, r]` hợp lệ với một `r < j*` — khi đó có cửa sổ ngắn hơn `L*`, mâu thuẫn tính tối ưu.

Vậy thuật toán ghi nhận được `L*`; và mọi giá trị nó ghi nhận đều là độ dài cửa sổ hợp lệ thật → kết quả đúng bằng `L*`. Trường hợp đã cân bằng được trả `0` riêng.

## 11. Time / Space Complexity

Gọi `n = len(s)`, bảng chữ cái `Σ = 4`.

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi xâu con | `O(n³)` | `O(n)` |
| 2. Prefix count | `O(n² · Σ)` | `O(n · Σ)` |
| 3. Nhị phân + cửa sổ cố định | `O(n · Σ · log n)` | `O(Σ)` |
| 4. Hai con trỏ | `O(n · Σ)` | `O(Σ)` |
| 5. Hai con trỏ + `missing` | `O(n)` | `O(Σ)` |

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [14]:
methods = {
    "brute": balanced_brute,
    "prefix_n2": balanced_prefix_n2,
    "binary_search": balanced_binary_search,
    "sliding": balanced_sliding,
    "need": balanced_need,
    "submit": lambda s: Solution().balancedString(s),
}
edge = [
    ("QQQQ", 3),
    ("WWEQERQWQWWRWWERQWEQ", 4),       # ví dụ cũ trên LeetCode
    ("QWERQWER", 0),
    ("EEEQQWWW", 4),                    # dư E và W ở hai đầu -> phải ôm cả "EQQW"
    ("RRRRRRRR", 6),
    ("QQWWEERR", 0),
]
for name, f in methods.items():
    for s, want in examples + edge:
        assert f(s) == want, (name, s, f(s))
print("Mọi cách đúng trên ví dụ + edge case")

Mọi cách đúng trên ví dụ + edge case


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

In [15]:
rng = random.Random(1234)
for _ in range(400):
    n = 4 * rng.randint(1, 5)
    s = "".join(rng.choice("QWER") for _ in range(n))
    want = balanced_brute(s)
    for name, f in methods.items():
        assert f(s) == want, (name, s)
for _ in range(200):
    n = 4 * rng.randint(5, 50)
    s = "".join(rng.choice(rng.choice(["QWER", "QQWER", "QQQQWE"])) for _ in range(n))
    want = balanced_prefix_n2(s)
    for name in ["binary_search", "sliding", "need", "submit"]:
        assert methods[name](s) == want, name
print("600 test ngẫu nhiên: mọi cách khớp trọng tài")

600 test ngẫu nhiên: mọi cách khớp trọng tài


### 12.3 Test giới hạn `n = 10^5`

In [16]:
n = 10 ** 5
big_cases = {
    "toàn Q": "Q" * n,
    "ngẫu nhiên lệch": "".join(rng.choice("QQQWER") for _ in range(n)),
    "hai đầu dư": "E" * (n // 2) + "Q" * (n // 4) + "W" * (n // 4),
}
for label, s in big_cases.items():
    res = {}
    for name in ["binary_search", "sliding", "need", "submit"]:
        t0 = time.perf_counter()
        res[name] = methods[name](s)
        res[name + "_ms"] = (time.perf_counter() - t0) * 1000
    assert len({res[k] for k in ["binary_search", "sliding", "need", "submit"]}) == 1
    print(f"{label:16s} -> {res['submit']:>6} | " + ", ".join(f"{k}: {res[k + '_ms']:.0f}ms" for k in ["binary_search", "sliding", "need", "submit"]))
assert Solution().balancedString("Q" * n) == 3 * n // 4
assert Solution().balancedString(big_cases["hai đầu dư"]) == n // 4

toàn Q           ->  75000 | binary_search: 167ms, sliding: 58ms, need: 15ms, submit: 25ms


ngẫu nhiên lệch  ->  49615 | binary_search: 182ms, sliding: 60ms, need: 19ms, submit: 30ms


hai đầu dư       ->  25000 | binary_search: 407ms, sliding: 62ms, need: 17ms, submit: 29ms


Kiểm tra lại bằng lý luận: `"Q" * n` → cửa sổ phải ôm `3n/4` chữ `Q` dư. Chuỗi "hai đầu dư" có `E` dư `n/4` (tất cả ở nửa đầu) → cửa sổ ngắn nhất là `n/4` chữ `E` bất kỳ liền nhau.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Nghĩ đáp án = tổng số ký tự dư

In [17]:
def wrong_excess(s):
    k = len(s) // 4
    cnt = Counter(s)
    return sum(max(0, cnt[c] - k) for c in "QWER")


s = "EEEQQWWW"
print("sai :", wrong_excess(s))          # 2 (dư 1 E, 1 W)
print("đúng:", balanced_sliding(s))      # 4 — phải ôm liên tiếp "EQQW"

sai : 2
đúng: 4


**Vì sao sai:** tổng dư chỉ là **cận dưới**. Xâu con phải **liên tiếp**; các ký tự dư có thể nằm xa nhau, buộc cửa sổ ôm cả những ký tự "vô tội" ở giữa. **Sửa:** cửa sổ trượt.

### Lỗi 2 — Kiểm tra số đếm **trong** cửa sổ thay vì **ngoài** cửa sổ

In [18]:
def wrong_inside(s):
    n, k = len(s), len(s) // 4
    best = n
    for i in range(n + 1):
        for j in range(i, n + 1):
            inside = Counter(s[i:j])
            if all(inside[c] <= k for c in "QWER"):         # sai điều kiện
                if j - i > 0:
                    best = min(best, j - i)
    return best


for s in ["QQQW", "QWER"]:
    print(s, "sai:", wrong_inside(s), "| đúng:", balanced_sliding(s))

QQQW sai: 1 | đúng: 2
QWER sai: 1 | đúng: 0


**Vì sao sai:** thứ ta **không** thay được là phần **bên ngoài**; ràng buộc phải đặt lên nó. Đếm bên trong vô nghĩa (bên trong thay thoải mái). **Sửa:** khởi tạo `out = Counter(s)` và **trừ** khi ký tự vào cửa sổ.

### Lỗi 3 — Quên hoàn lại đếm khi co trái

In [19]:
def wrong_no_restore(s):
    n, k = len(s), len(s) // 4
    out = Counter(s)
    if all(out[c] <= k for c in "QWER"):
        return 0
    best, left = n, 0
    for right, ch in enumerate(s):
        out[ch] -= 1
        while left <= right and all(out[c] <= k for c in "QWER"):
            best = min(best, right - left + 1)
            left += 1                                   # quên out[s[left]] += 1
    return best


s = "WQWRQQQW"
print("sai :", wrong_no_restore(s))
print("đúng:", balanced_sliding(s))

sai : 1
đúng: 3


**Vì sao sai:** ký tự bị đẩy ra khỏi cửa sổ trở về "bên ngoài" nên `out` phải tăng. Quên bước này khiến `out` nhỏ hơn thực tế → cửa sổ "giả hợp lệ" ngắn hơn đáp án thật. **Sửa:** `out[s[left]] += 1` trước `left += 1`.

### Lỗi 4 — Không xử lý trường hợp đã cân bằng

In [20]:
def wrong_no_zero_check(s):
    n, k = len(s), len(s) // 4
    out = Counter(s)
    best, left = n, 0
    for right, ch in enumerate(s):
        out[ch] -= 1
        while left <= right and all(out[c] <= k for c in "QWER"):
            best = min(best, right - left + 1)
            out[s[left]] += 1
            left += 1
    return best


print("sai :", wrong_no_zero_check("QWER"))   # 1 — cửa sổ nhỏ nhất xét được dài 1
print("đúng:", balanced_sliding("QWER"))      # 0

sai : 1
đúng: 0


**Vì sao sai:** vòng lặp chỉ xét cửa sổ **khác rỗng** (`left ≤ right`), nên không bao giờ ra `0`. **Sửa:** kiểm tra cân bằng trước và trả `0` (hoặc cho phép cửa sổ rỗng).

## 14. Mẹo & nhận dạng pattern

- "**Xâu con / mảng con ngắn nhất** sao cho …" + điều kiện **đơn điệu khi mở rộng** → **cửa sổ trượt**: mở rộng phải, `while` hợp lệ thì co trái và cập nhật.
- "Thay đoạn bằng bất cứ thứ gì" → chuyển ràng buộc sang **phần còn lại** (bên ngoài cửa sổ).
- Bảng chữ cái nhỏ cố định (4 ký tự) → kiểm tra `all(...)` `O(4)` là đủ; bảng lớn → dùng biến đếm `missing`.
- Đơn điệu theo độ dài → luôn có phương án dự phòng **nhị phân + cửa sổ cố định** `O(n log n)`.
- Cùng họ: Minimum Window Substring (76), Minimum Size Subarray Sum (209), Max Consecutive Ones III (1004).

## 15. Biến thể

1. **Bảng chữ cái tổng quát** `m` ký tự, mỗi ký tự phải xuất hiện `n/m` lần — cùng thuật toán, kiểm tra `m` bộ đếm.
2. **Mỗi ký tự có số lượng mục tiêu riêng** `target[c]` (tổng = `n`) — điều kiện `out[c] ≤ target[c]`.
3. **Được thay `t` đoạn rời nhau** — khó hơn, thường cần DP.
4. **Đếm số xâu con hợp lệ** (không chỉ ngắn nhất): với mỗi `left`, mọi `right ≥ right_min(left)` đều hợp lệ → cộng `n − right_min(left)`.
5. **Minimum Window Substring (LeetCode 76):** cửa sổ ngắn nhất chứa mọi ký tự của `t` (có bội).

In [21]:
def min_window_targets(s, target):
    """Biến thể 2: mỗi ký tự c phải xuất hiện đúng target[c] lần sau khi thay (sum(target) == len(s))."""
    n = len(s)
    out = Counter(s)
    if all(out[c] <= target.get(c, 0) for c in out):
        return 0
    best, left = n, 0
    for right, ch in enumerate(s):
        out[ch] -= 1
        while left <= right and all(out[c] <= target.get(c, 0) for c in out):
            best = min(best, right - left + 1)
            out[s[left]] += 1
            left += 1
    return best


print(min_window_targets("aaab", {"a": 2, "b": 2}))       # thay 1 'a' -> 'b'
print(min_window_targets("abcabc", {"a": 1, "b": 1, "c": 4}))

1
2


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi xâu con | kiểm tra mọi `(i, j)` | `O(n³)` | `O(n)` | trọng tài |
| 2. Prefix count | đếm `O(1)`, `j` nhỏ nhất cho mỗi `i` | `O(n²)` | `O(n)` | hiểu điều kiện |
| 3. Nhị phân + cửa sổ cố định | `can(L)` đơn điệu | `O(n log n)` | `O(1)` | khi chưa nghĩ ra hai con trỏ |
| 4. Hai con trỏ | mở rộng phải, co trái | `O(n)` | `O(1)` | **nộp bài** |
| 5. Hai con trỏ + missing | dạng Minimum Window | `O(n)` | `O(1)` | bảng chữ lớn, khuôn mẫu chung |

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test.

### Bài 1 (đoán output)

`Solution().balancedString("QQQQWWWW")` = ?

### Bài 2 (tính tay)

Với `s = "WQWRQQQW"` (`k = 2`): đếm toàn bộ, tính lượng dư, rồi tìm cửa sổ ngắn nhất.

### Bài 3 (điền TODO) — đếm số xâu con hợp lệ

In [22]:
def count_valid_windows(s):
    """Đếm số cặp (i, j), 0 <= i < j <= n, sao cho thay s[i:j] làm s cân bằng (chỉ xét cửa sổ khác rỗng)."""
    n, k = len(s), len(s) // 4
    out = Counter(s)
    total, right = 0, 0          # right: cửa sổ hiện tại là s[left:right]
    for left in range(n):
        # TODO: tăng right (đưa s[right] vào) đến khi s[left:right] hợp lệ hoặc right == n
        #       nếu hợp lệ: cộng n - right + 1 (mọi j từ right..n), rồi đưa s[left] ra trước khi sang left+1
        raise NotImplementedError
    return total

# print(count_valid_windows("QQQW"))   # mong đợi 5: [0:2],[0:3],[0:4],[1:3],[1:4]

### Bài 4 (tự viết lại)

Viết lại lời giải để trả về **chính cửa sổ** `(i, j)` ngắn nhất (nửa mở `s[i:j]`), nếu hoà lấy cửa sổ **bắt đầu sớm nhất**. Với `s` đã cân bằng trả `(0, 0)`.

In [23]:
def best_window_todo(s):
    # TODO
    raise NotImplementedError

# print(best_window_todo("QQQW"))   # mong đợi (0, 2)

### Bài 5 (mini challenge) — bảng chữ cái tổng quát

Viết `balanced_general(s, alphabet)` với `len(s)` chia hết cho `len(alphabet)`, mỗi ký tự cần xuất hiện `len(s) // len(alphabet)` lần. Yêu cầu `O(n)` không phụ thuộc kích thước bảng chữ (dùng biến đếm số ký tự đang vượt mức).

In [24]:
def balanced_general_todo(s, alphabet):
    # TODO: over = số ký tự c có out[c] > k; cửa sổ hợp lệ <=> over == 0
    raise NotImplementedError

# print(balanced_general_todo("aaabbc", "abc"))   # mong đợi 1 (thay một 'a' -> 'c')

## 18. Gợi ý 3 mức

**Bài 1.** Mức 1: `k = 2`, `Q` dư 2, `W` dư 2. Mức 2: cửa sổ phải chứa ≥ 2 `Q` và ≥ 2 `W`. Mức 3: ngắn nhất `"QQWW"` → `4`.

**Bài 2.** Mức 1: `W=3, Q=4, R=1, E=0`. Mức 2: dư `W1, Q2`. Mức 3: cửa sổ `"QQW"` ở cuối (chỉ số 5..7) dài `3`.

**Bài 3.** Mức 1: `right` không bao giờ lùi (đơn điệu). Mức 2: `while right < n and not valid: out[s[right]] -= 1; right += 1`. Mức 3: nếu `valid` thì `total += n − right + 1`; trước khi tăng `left`, nếu `left < right` thì `out[s[left]] += 1`, ngược lại `right += 1` (cửa sổ rỗng thì giữ `right ≥ left + 1`).

**Bài 4.** Mức 1: lưu thêm `best_i`. Mức 2: chỉ cập nhật khi độ dài **nhỏ hơn chặt**. Mức 3: trả `(best_i, best_i + best)`.

**Bài 5.** Mức 1: `over` khởi tạo = số ký tự có `out[c] > k`. Mức 2: khi `out[c]` giảm từ `k+1` xuống `k` thì `over −= 1`; tăng từ `k` lên `k+1` thì `over += 1`. Mức 3: `while over == 0` thì co trái.

## 19. Đáp án tham khảo

In [25]:
# Bài 1, 2
print(Solution().balancedString("QQQQWWWW"))
assert Solution().balancedString("QQQQWWWW") == 4
print(Counter("WQWRQQQW"), Solution().balancedString("WQWRQQQW"))
assert Solution().balancedString("WQWRQQQW") == 3

4
Counter({'Q': 4, 'W': 3, 'R': 1}) 3


In [26]:
def is_valid(out, k):
    return all(out[c] <= k for c in "QWER")


def count_valid_windows(s):
    n, k = len(s), len(s) // 4
    out = Counter(s)
    total, right = 0, 0
    for left in range(n):
        if right == left:                    # cửa sổ rỗng -> đưa ít nhất một ký tự vào
            out[s[right]] -= 1
            right += 1
        while right < n and not is_valid(out, k):
            out[s[right]] -= 1
            right += 1
        if is_valid(out, k):
            total += n - right + 1
        out[s[left]] += 1                     # s[left] ra khỏi cửa sổ
    return total


def best_window(s):
    n, k = len(s), len(s) // 4
    out = Counter(s)
    if is_valid(out, k):
        return (0, 0)
    best, best_i, left = n + 1, 0, 0
    for right, ch in enumerate(s):
        out[ch] -= 1
        while left <= right and is_valid(out, k):
            if right - left + 1 < best:
                best, best_i = right - left + 1, left
            out[s[left]] += 1
            left += 1
    return (best_i, best_i + best)


def balanced_general(s, alphabet):
    n, k = len(s), len(s) // len(alphabet)
    out = Counter(s)
    over = sum(1 for c in alphabet if out[c] > k)
    if over == 0:
        return 0
    best, left = n, 0
    for right, ch in enumerate(s):
        out[ch] -= 1
        if out[ch] == k:
            over -= 1
        while over == 0:
            best = min(best, right - left + 1)
            lc = s[left]
            out[lc] += 1
            if out[lc] == k + 1:
                over += 1
            left += 1
    return best


def brute_count(s):
    n = len(s)
    return sum(valid_window(s, i, j) for i in range(n) for j in range(i + 1, n + 1))


def brute_general(s, alphabet):
    n, k = len(s), len(s) // len(alphabet)
    best = n
    for i in range(n + 1):
        for j in range(i, n + 1):
            out = Counter(s[:i] + s[j:])
            if all(out[c] <= k for c in alphabet):
                best = min(best, j - i)
    return best


assert count_valid_windows("QQQW") == 5
assert best_window("QQQW") == (0, 2)
assert balanced_general("aaabbc", "abc") == 1
rng = random.Random(4321)
for _ in range(300):
    n = 4 * rng.randint(1, 4)
    s = "".join(rng.choice("QWER") for _ in range(n))
    assert count_valid_windows(s) == brute_count(s), s
    i, j = best_window(s)
    assert j - i == balanced_brute(s) and (i == j or valid_window(s, i, j))
    alpha = rng.choice(["ab", "abc", "QWER", "abcdef"])
    m = len(alpha) * rng.randint(1, 3)
    g = "".join(rng.choice(alpha) for _ in range(m))
    assert balanced_general(g, alpha) == brute_general(g, alpha), (g, alpha)
print("Đáp án bài 3, 4, 5 đúng")

Đáp án bài 3, 4, 5 đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
k = n // 4
out = Counter(s)                          # mọi thứ ban đầu ở "ngoài"
if all(out[c] <= k for c in "QWER"): return 0
ans, left = n, 0
for right, ch in enumerate(s):
    out[ch] -= 1                          # vào cửa sổ
    while left <= right and all(out[c] <= k for c in "QWER"):
        ans = min(ans, right - left + 1)
        out[s[left]] += 1                 # ra khỏi cửa sổ — đừng quên!
        left += 1
return ans
```

**Checklist:**
- [ ] Điều kiện đặt lên **bên ngoài** cửa sổ (`out[c] ≤ n/4`)?
- [ ] Trả `0` khi đã cân bằng?
- [ ] Co trái thì `out[s[left]] += 1`?
- [ ] `while left <= right` để không co quá cửa sổ rỗng?
- [ ] Không dùng "tổng số ký tự dư" làm đáp án?
- [ ] Thử: `"QWER"`, `"QQQQ"`, ký tự dư ở hai đầu (`"EEEQQWWW"`)?